## 06 Buildings

**笔记本**：`06 buildings20260403.1.ipynb`

**库**：`geopandas`、`pandas`、`numpy`、`matplotlib`、`shapely`（`box`、`polylabel` 等）等。

**输入**：
- `data_raw/Shenzen_Buildings_General/Shenzen_Buildings_General.shp`（普查建筑）
- `../04 Transport/data_raw/shenzhen_boundary.geojson`
- `../00/16-built-up-area/shenzhen_built_up_area_2020.shp`（建成区，若用于下游）

**做了什么 / 算了什么**：
- 清洗：加载建筑普查数据，并筛“已竣工”建筑；修复无效几何，去掉坏数据；
竣工状态 BLDCOND == "已竣"、make_valid、去掉空几何；AREAF >= 10、BLDG_HEIGH >= 0；裁剪到深圳。
- 字段英文化；
- 派生建筑级别属性：高度分档 height_class、超高 is_tall（>100 m）可以作为高摩擦或高活动区的 proxy 之一、用途映射 usage_cat、volume_m3 = 占地 × 高度，同时考虑横向规模、纵向规模
- 内点深度代理： 对抽样建筑用 polylabel 等（注意需在投影坐标下才表示真实米；否则数值可能接近 0）。
（polylabel 通常会找到一个 polygon 内部“最居中”的点，再去量这个点到建筑边界的距离就得到一个建筑的“内深度”近似，建筑从外缘到内部核心的进入深度）
- 在深圳边界内使用 **H3 res=8**：**优先**读取 `../03 Boundary/data_out/sz_hex_grid_res8.gpkg`（与 `03b hex_grid` 完全一致），否则本地 `geo_to_cells` 回退；建筑 centroid 与格 `sjoin…within` 归属。
- 按格汇总： 栋数、高度 mean/max/std、占地与体积合计、平均层数、超高栋数、建筑密度 / 体积密度（用六边形在 EPSG:32650 下的真实面积 m²）、主导用途 dominant_usage、用途分布的 Shannon 熵 → usage_diversity。
- 建筑质心是否在 2020 建成区内 的比例统计。

**写出文件**：
- `data_out/sz_buildings.gpkg`
- `data_out/sz_building_grid.gpkg`

**典型输出信息**：原始/清洗栋数、高度与占地统计、高度分级与用途分布、网格单元数与保存路径。
- 做 Ground Friction 的建筑形态部分,识别“大型封闭综合体/园区型空间”(地面最后一段摩擦高的空间), 解释 Demand Pressure / Overload


In [2]:
# ============================================================
#  06 Buildings / Morphology
#  数据源: 普查版本 (573K 栋, 带高度/层数/用途)
#  输出: 逐栋建筑 + 网格化形态指标
# ============================================================

from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, mapping
import h3
import warnings
warnings.filterwarnings("ignore")

RAW = Path("data_raw")
OUT = Path("data_out")
OUT.mkdir(exist_ok=True)

SHP = RAW / "Shenzen_Buildings_General/Shenzen_Buildings_General.shp"
BOUNDARY = Path("../04 Transport/data_raw/shenzhen_boundary.geojson")
BUILT_UP_2020 = Path("../00/16-built-up-area/shenzhen_built_up_area_2020.shp")

# ── 加载深圳边界 ──
shenzhen = gpd.read_file(BOUNDARY).to_crs(4326)
shenzhen_geom = shenzhen.union_all() if hasattr(shenzhen, "union_all") else shenzhen.unary_union

# ── 加载普查建筑 ──
print("Loading census buildings ...")
bldg_raw = gpd.read_file(SHP)
print(f"  Raw: {len(bldg_raw):,} buildings")

# ── 清洗 ──
bldg = bldg_raw.copy()

# 1) 只保留已竣工
bldg = bldg[bldg["BLDCOND"] == "已竣"]

# 2) 修复无效几何
bldg["geometry"] = bldg.geometry.make_valid()
bldg = bldg[~bldg.geometry.is_empty & bldg.geometry.notna()]

# 3) 过滤异常值: 面积 < 10 m² 或高度 < 0
bldg = bldg[(bldg["AREAF"] >= 10) & (bldg["BLDG_HEIGH"] >= 0)]

# 4) 裁剪到深圳边界
bldg = gpd.clip(bldg, shenzhen_geom)

# 5) 重命名列, 统一英文字段名
bldg = bldg.rename(columns={
    "BLDG_HEIGH": "height_m",
    "UP_BLDG_FL": "floors",
    "DOWN_BLDG_": "basement_floors",
    "AREAF": "footprint_m2",
    "BLDG_USAGE": "usage",
    "BLDSTRU": "structure",
    "NOWNAME": "name",
    "BLDADDR": "address",
})

# 6) 保留有用列
keep_cols = ["name", "address", "usage", "structure", "height_m", "floors",
             "basement_floors", "footprint_m2", "geometry"]
bldg = bldg[[c for c in keep_cols if c in bldg.columns]]

bldg = bldg.reset_index(drop=True)
print(f"  Cleaned: {len(bldg):,} buildings")
print(f"  Height: mean={bldg['height_m'].mean():.1f}m, max={bldg['height_m'].max():.0f}m")
print(f"  Area: mean={bldg['footprint_m2'].mean():.0f} m², median={bldg['footprint_m2'].median():.0f} m²")

Loading census buildings ...
  Raw: 573,007 buildings
  Cleaned: 567,309 buildings
  Height: mean=12.4m, max=442m
  Area: mean=1937 m², median=468 m²


In [3]:
# ============================================================
#  1. 逐栋派生指标
# ============================================================

# 高度分类 (对无人机飞行高度管制有意义)
bldg["height_class"] = pd.cut(
    bldg["height_m"],
    bins=[0, 3, 10, 24, 50, 100, 999],
    labels=["<3m", "3-10m", "10-24m", "24-50m", "50-100m", ">100m"],
    right=True,
)

bldg["is_tall"] = bldg["height_m"] > 100  # 超高层

# 用途大类映射 (原始分类太细, 归为大类便于可视化)
USAGE_MAP = {
    "私宅": "residential", "居住": "residential",
    "居住配套": "residential_support",
    "商服": "commercial", "办公": "commercial",
    "工业": "industrial", "仓储": "industrial",
    "公共设施": "public", "公共配套": "public",
    "交通": "transport", "市政": "public",
    "教育": "public", "医疗": "public",
}
bldg["usage_cat"] = bldg["usage"].map(USAGE_MAP).fillna("other")

# 体积近似 (footprint × height), 用于建筑密度计算
bldg["volume_m3"] = bldg["footprint_m2"] * bldg["height_m"]

print("=== Height Class Distribution ===")
print(bldg["height_class"].value_counts().sort_index().to_string())
print(f"\nSuper-tall (>100m): {bldg['is_tall'].sum()}")
print(f"\n=== Usage Category ===")
print(bldg["usage_cat"].value_counts().to_string())

=== Height Class Distribution ===
height_class
<3m        150576
3-10m      145125
10-24m     234906
24-50m      28842
50-100m      6884
>100m         914

Super-tall (>100m): 914

=== Usage Category ===
usage_cat
residential            402059
industrial             107073
commercial              21793
residential_support     18902
public                  12797
other                    3091
transport                1594


In [4]:
# ============================================================
#  2. 网格化形态指标 — H3 res 8（优先读 03b 导出，与全仓库 h3_id 一致）
#  建筑 centroid → sjoin within 六边形
# ============================================================

H3_RES = 8
GRID_03_R8 = Path("../03 Boundary/data_out/sz_hex_grid_res8.gpkg")


def _h3_to_polygon(h: str) -> Polygon:
    coords = h3.cell_to_boundary(h)
    return Polygon([(lng, lat) for lat, lng in coords])


if GRID_03_R8.exists():
    grid = gpd.read_file(GRID_03_R8)
    if "h3_id" not in grid.columns:
        raise ValueError(f"{GRID_03_R8} 须含列 h3_id")
    keep = [c for c in ("h3_id", "geometry") if c in grid.columns]
    grid = grid[keep].copy()
    grid["h3_id"] = grid["h3_id"].astype(str)
    grid["h3_res"] = H3_RES
    print(f"Grid: {len(grid):,} H3 cells from 03b ({GRID_03_R8.name})")
else:
    poly_gj = mapping(shenzhen_geom)
    hex_ids = sorted(h3.geo_to_cells(poly_gj, res=H3_RES))
    grid = gpd.GeoDataFrame(
        {"h3_id": hex_ids, "h3_res": H3_RES},
        geometry=[_h3_to_polygon(h) for h in hex_ids],
        crs=4326,
    )
    print(f"Grid: {len(grid):,} H3 cells (fallback geo_to_cells res={H3_RES}) — 建议先运行 03b")

grid_metric = grid.to_crs(32650)
grid["cell_area_m2"] = grid_metric.geometry.area.values

# 空间连接: 建筑 → 网格 (用建筑质心)
bldg_centroids = bldg.copy()
bldg_centroids["geometry"] = bldg_centroids.geometry.centroid
joined = gpd.sjoin(bldg_centroids, grid[["h3_id", "geometry"]], how="inner", predicate="within")

# 按网格汇总
grid_stats = joined.groupby("h3_id").agg(
    building_count=("height_m", "count"),
    avg_height=("height_m", "mean"),
    max_height=("height_m", "max"),
    height_std=("height_m", "std"),
    total_footprint=("footprint_m2", "sum"),
    total_volume=("volume_m3", "sum"),
    avg_floors=("floors", "mean"),
    n_tall=("is_tall", "sum"),
).reset_index()

areas = grid[["h3_id", "cell_area_m2"]]
grid_stats = grid_stats.merge(areas, on="h3_id", how="left")
grid_stats["building_density"] = grid_stats["total_footprint"] / grid_stats["cell_area_m2"]
grid_stats["volume_density"] = grid_stats["total_volume"] / grid_stats["cell_area_m2"]

# 主导用途 (众数)
dominant = joined.groupby("h3_id")["usage_cat"].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else "other"
)
dominant.name = "dominant_usage"
grid_stats = grid_stats.merge(dominant, on="h3_id", how="left")

# 填充无建筑的网格
grid_stats["height_std"] = grid_stats["height_std"].fillna(0)

# 合并回网格几何
grid_result = grid.merge(grid_stats, on="h3_id", how="left")
grid_result = grid_result.fillna(0)
grid_result.loc[grid_result["dominant_usage"] == 0, "dominant_usage"] = "empty"

# 用途多样性 (Shannon 熵)
from scipy.stats import entropy as shannon_entropy

usage_pivot = joined.groupby(["h3_id", "usage_cat"]).size().unstack(fill_value=0)
diversity = usage_pivot.apply(lambda r: shannon_entropy(r[r > 0]), axis=1)
diversity.name = "usage_diversity"
grid_stats = grid_stats.merge(diversity, on="h3_id", how="left")
grid_stats["usage_diversity"] = grid_stats["usage_diversity"].fillna(0)

# 合并回网格几何
grid_result = grid.merge(grid_stats, on="h3_id", how="left")
grid_result = grid_result.fillna(0)
grid_result.loc[grid_result["dominant_usage"] == 0, "dominant_usage"] = "empty"

print(f"Grids with buildings: {(grid_result['building_count'] > 0).sum():,} / {len(grid_result):,}")
print(f"Avg building density: {grid_result.loc[grid_result['building_count'] > 0, 'building_density'].mean():.3f}")
print(f"Max height in any grid: {grid_result['max_height'].max():.0f}m")
print(f"Usage diversity: mean={grid_result.loc[grid_result['building_count'] > 0, 'usage_diversity'].mean():.3f}")


Grid: 2,754 H3 cells from 03b (sz_hex_grid_res8.gpkg)
Grids with buildings: 2,237 / 2,754
Avg building density: 0.680
Max height in any grid: 442m
Usage diversity: mean=0.792


In [5]:
# ============================================================
#  3. 建成区校验 + 保存输出 + 统计摘要
# ============================================================

# ── 建成区 2020 校验 ──
if BUILT_UP_2020.exists():
    built_up = gpd.read_file(BUILT_UP_2020).to_crs(4326)
    built_up_geom = built_up.union_all() if hasattr(built_up, "union_all") else built_up.unary_union
    from shapely.prepared import prep as _prep
    bu_prep = _prep(built_up_geom)
    in_built_up = bldg.geometry.centroid.apply(lambda pt: bu_prep.contains(pt))
    bldg["in_built_up_area"] = in_built_up
    print(f"Built-up area check: {in_built_up.sum():,} / {len(bldg):,} buildings inside ({in_built_up.mean()*100:.1f}%)")
    # 也标注到网格
    grid_result["in_built_up_area"] = grid_result.geometry.centroid.apply(lambda pt: bu_prep.contains(pt))
    del built_up, built_up_geom, bu_prep
else:
    bldg["in_built_up_area"] = True
    grid_result["in_built_up_area"] = True
    print("Built-up area shp not found, skipping")

# 逐栋建筑
bldg.to_file(OUT / "sz_buildings.gpkg", driver="GPKG")
print(f"Buildings -> data_out/sz_buildings.gpkg  ({len(bldg):,} rows)")

# 网格化形态
grid_result.to_file(OUT / "sz_building_grid.gpkg", driver="GPKG")
print(f"Grid      -> data_out/sz_building_grid.gpkg  ({len(grid_result):,} cells)")

# ── 统计摘要 ──
print("\n=== Building Summary ===")
print(f"Total buildings: {len(bldg):,}")
print(f"Height: mean={bldg['height_m'].mean():.1f}m  median={bldg['height_m'].median():.1f}m  max={bldg['height_m'].max():.0f}m")
print(f"Floors: mean={bldg['floors'].mean():.1f}  max={bldg['floors'].max():.0f}")
print(f"Super-tall (>100m): {bldg['is_tall'].sum()}")

print("\n=== Height Distribution ===")
print(bldg["height_class"].value_counts().sort_index().to_string())

print("\n=== Usage Distribution ===")
print(bldg["usage_cat"].value_counts().to_string())

print("\n=== Grid Morphology (non-empty grids) ===")
g = grid_result[grid_result["building_count"] > 0]
print(f"Building density: mean={g['building_density'].mean():.3f}  max={g['building_density'].max():.3f}")
print(f"Volume density:   mean={g['volume_density'].mean():.1f}  max={g['volume_density'].max():.1f}")
print(f"Dominant usage distribution:")
print(g["dominant_usage"].value_counts().to_string())

Built-up area check: 546,949 / 567,309 buildings inside (96.4%)
Buildings -> data_out/sz_buildings.gpkg  (567,309 rows)
Grid      -> data_out/sz_building_grid.gpkg  (2,754 cells)

=== Building Summary ===
Total buildings: 567,309
Height: mean=12.4m  median=9.0m  max=442m
Floors: mean=4.1  max=115
Super-tall (>100m): 914

=== Height Distribution ===
height_class
<3m        150576
3-10m      145125
10-24m     234906
24-50m      28842
50-100m      6884
>100m         914

=== Usage Distribution ===
usage_cat
residential            402059
industrial             107073
commercial              21793
residential_support     18902
public                  12797
other                    3091
transport                1594

=== Grid Morphology (non-empty grids) ===
Building density: mean=0.680  max=4.419
Volume density:   mean=27.9  max=631.2
Dominant usage distribution:
dominant_usage
residential            1437
industrial              546
public                  116
commercial               83
ot